# 👤 Siddarth's Personal RAG Assistant
This notebook contains your personal details, skills, and projects. It uses **RAG (Retrieval-Augmented Generation)** so you can ask it questions like *"What are my skills?"* and it will answer based strictly on the provided data.

### 1. Install Dependencies
We'll use `sentence-transformers` for creating text embeddings, `faiss-cpu` for the vector database, and `google-generativeai` for the LLM.

In [ ]:
!pip install -q sentence-transformers faiss-cpu google-generativeai

### 2. Enter Your Gemini API Key
Get your free key from [Google AI Studio](https://aistudio.google.com/).

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# It's best practice to store this in Colab Secrets (the key icon on the left)
# Name the secret 'GEMINI_API_KEY'. If you haven't, you can just paste it below temporarily.
try:
    api_key = userdata.get('GEMINI_API_KEY')
except:
    api_key = input("Enter your Gemini API Key: ")

genai.configure(api_key=api_key)
llm = genai.GenerativeModel('gemini-2.5-flash')

### 3. Your Personal Data (The Knowledge Base)
I've pre-filled this with data based on what we've worked on! Feel free to edit or add more.

In [ ]:
portfolio_data = [
    "Name: Siddarth Patel Kama",
    "Summary: A passionate software developer specializing in AI, Large Language Models (LLMs), and modern web development.",
    "Skills: React, Next.js, Python, JavaScript, TypeScript, TailwindCSS.",
    "AI Skills: RAG (Retrieval-Augmented Generation), LLM Fine-tuning (QLoRA, Unsloth), Prompt Engineering, Vector Databases (FAISS, Supabase pgvector).",
    "Project 1: ScamShield - An 'Evidence Engine' RAG application that verifies internship offer letters against official corporate policies to detect scams.",
    "Project 2: BYOK Chatbot - A custom Next.js multimodal chat interface that allows users to bring their own API keys.",
    "Project 3: Friendly Macha AI - Fine-tuned the Qwen 2.5 3B model using QLoRA and Unsloth to act as a highly conversational, slang-fluent coding assistant."
]

print(f"Loaded {len(portfolio_data)} data chunks about Siddarth.")

### 4. Create Embeddings & Vector Database
This turns your text into numbers (vectors) so the system can search through them.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1. Load a lightweight embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Convert text to embeddings
embeddings = embedder.encode(portfolio_data)

# 3. Create FAISS index (Vector Database)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Vector Database created successfully!")

### 5. Start the Live Chat!
Run this cell to open an interactive chat interface right here in the notebook. It will remember previous questions within this session.

In [ ]:
print("\033[92m\n\ud83e\udd16 Welcome to Siddarth's Portfolio Assistant! (Type 'quit' to exit)\033[0m")
print("-" * 60)

# Start a chat session to keep memory of the conversation
chat = llm.start_chat(history=[])

while True:
    question = input("\n\033[94mYou:\033[0m ")
    if question.lower() in ['quit', 'exit', 'stop', 'bye']:
        print("\n\033[92mSiddarthBot: See ya!\033[0m")
        break
        
    if not question.strip():
        continue

    # Step 1: Embed the question
    question_embedding = embedder.encode([question])
    
    # Step 2: Search FAISS for the top 3 most relevant facts
    D, I = index.search(question_embedding, k=3)
    
    # Step 3: Gather the retrieved context
    retrieved_context = "\n".join([portfolio_data[i] for i in I[0]])
    
    # Step 4: Ask Gemini using the context
    prompt = f"""
    You are an AI assistant representing Siddarth Patel Kama.
    Answer the user's question based strictly on the Context provided below.
    If the answer is not in the context, politely say that you don't have that information about Siddarth.
    Keep answers concise, professional, and friendly.
    
    Context:
    {retrieved_context}
    
    User Question: {question}
    """
    
    response = chat.send_message(prompt)
    
    # Print the answer
    print("\033[92m\nSiddarthBot:\033[0m " + response.text)
    
    # Optional: Print the hidden context it used to show off the RAG engine
    print("\n\033[90m(RAG Engine used these chunks: " + ", ".join([str(i) for i in I[0]]) + ")\033[0m")
